# CohMetrix Grammar Pipeline

Prepares speech texts for CohMetrix, automates the web upload, and merges the resulting
syntactic/readability indices back into the speech dataset.

**Three steps in this notebook:**
1. **Text preparation** — filter Trump 2020 press briefings, clean and truncate each speech,
   save one `.txt` file per speech in `processed_texts/`
2. **CohMetrix web automation** — Selenium + Tesseract OCR to solve the CAPTCHA and submit texts
3. **Merge output** — read all `CohMetrixOutput (N).txt` files and merge with the speech dataset

**Requirements for Step 2:** ChromeDriver matching your Chrome version, Tesseract OCR installed,
and the Memphis University CohMetrix server reachable at `http://141.225.61.35/CohMetrix2017/`

In [ ]:
!pip install -q selenium pytesseract opencv-python pillow pandas

import os
import re
import time
from io import BytesIO

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance

DATA_PATH       = "4_Data_with_AI_Market_Indicies.csv"
OUTPUT_FOLDER   = "processed_texts"
COHMETRIX_FOLDER = "Resources/Trump_full/"
OUTPUT_MERGED   = "Data/6_TrumpwithCohmetrix.csv"
MAX_TEXT_LENGTH = 15000

## Step 1 — Text Preparation for CohMetrix

Filters Trump 2020 press briefings, strips non-standard characters with `limit_text()`,
truncates to 15 000 characters (CohMetrix hard limit), and saves one `.txt` file per speech.

File names use the original DataFrame index so they can be traced back to the dataset.

In [ ]:
def limit_text(text: str, max_length: int = MAX_TEXT_LENGTH) -> str:
    """Strip non-standard characters (preserving punctuation) and truncate."""
    cleaned = re.sub(r'[^a-zA-Z0-9\s\.,;?!\'"\(\)\[\]\{\}<>]', "", text)
    return cleaned[:max_length]


df = pd.read_csv(DATA_PATH, sep=",")
df["Date Time"] = pd.to_datetime(df["Date Time"], format="%Y-%m-%d %H:%M:%S%z")

trump_2020 = df[
    (df["Date Time"].dt.year == 2020)
    & (df["Date Time"].dt.month < 9)
    & (df["Title"].str.contains("Press Bri"))
    & (df["Administration"].str.contains("Trump"))
].copy()

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

for index, row in trump_2020.iterrows():
    processed = limit_text(row["Main text"])
    with open(os.path.join(OUTPUT_FOLDER, f"{index}.txt"), "w", encoding="utf-8") as f:
        f.write(processed)

print(f"Saved {len(trump_2020)} text files to '{OUTPUT_FOLDER}/'")

## Step 2 — CohMetrix Web Automation

Uploads each text file to the CohMetrix web interface at the Memphis University server.

**How it works:**
- Opens Chrome via Selenium and navigates to the CohMetrix page
- Pastes each speech text into the text area
- Uses Tesseract OCR + OpenCV image processing to extract and enter the CAPTCHA
- If CAPTCHA fails, resets and retries; if the server returns an error page, waits 20 s and retries
- Downloads results by clicking "Save Data"

**This step requires human supervision** — the server is frequently unreliable.
Run `check_ping()` first to confirm the server is reachable.

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import NoSuchElementException
import pytesseract
import subprocess


def check_ping(host: str = "141.225.61.35") -> bool:
    """Returns True when the CohMetrix server is reachable."""
    try:
        subprocess.check_output(["ping", "-c", "1", host], stderr=subprocess.STDOUT)
        print("Server reachable.")
        return True
    except subprocess.CalledProcessError:
        print("Server unreachable.")
        return False


def solve_captcha(browser) -> str:
    """
    Screenshots the page, crops the CAPTCHA region, runs OCR, and returns the extracted text.
    Coordinates (left, top, right, bottom) assume a maximised Chrome window on a standard display.
    """
    screenshot = Image.open(BytesIO(browser.get_screenshot_as_png()))
    captcha_img = screenshot.crop((294, 900, 891, 1050))
    captcha_img.save("captcha.png")

    img = cv2.imread("captcha.png", cv2.IMREAD_GRAYSCALE)
    _, binary = cv2.threshold(img, 180, 255, cv2.THRESH_BINARY)
    kernel  = np.ones((3, 3), np.uint8)
    cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    cleaned = np.array(ImageEnhance.Contrast(Image.fromarray(cleaned)).enhance(2))
    cleaned = cv2.erode(cleaned, kernel, iterations=1)
    cleaned = cv2.dilate(cleaned, kernel, iterations=1)

    gray   = cv2.cvtColor(cv2.imread("captcha.png"), cv2.COLOR_BGR2GRAY)
    blur   = cv2.GaussianBlur(gray, (3, 3), 0)
    thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    kernel2  = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    opening  = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel2, iterations=1)
    invert   = 255 - opening

    text = pytesseract.image_to_string(
        invert,
        config="--psm 10 --oem 3 -c tessedit_char_whitelist="
               "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz",
    )
    return text.strip()


def run_cohmetrix(list_text: list, base_url: str = "http://141.225.61.35/CohMetrix2017/") -> None:
    browser = webdriver.Chrome()
    browser.set_page_load_timeout(30000)
    browser.maximize_window()

    for count, text in enumerate(list_text):
        print(f"Processing speech {count + 1}/{len(list_text)} ...")
        text = limit_text(text, MAX_TEXT_LENGTH)
        uploaded = False

        while not uploaded:
            browser.get(base_url)
            time.sleep(5)
            browser.find_element("id", "Text").send_keys(text)

            captcha_text = ""
            while not captcha_text:
                captcha_text = solve_captcha(browser)
                print(f"  CAPTCHA: {captcha_text!r}")
                browser.find_element("id", "CaptchaText").send_keys(captcha_text)
                try:
                    browser.find_element(
                        "xpath",
                        '//*[@style="border: 0px solid rgb(141, 27, 27); '
                        'width:300px;color:red;padding: 5px;"]',
                    )
                    captcha_text = ""  # validation failed — retry
                except NoSuchElementException:
                    print("  CAPTCHA accepted.")
                    browser.find_element("name", "Savedata").click()
                    time.sleep(3)

            try:
                browser.find_element("xpath", "//title[contains(text(), 'Error')]")
                print("  Server error — retrying in 20 s ...")
                time.sleep(20)
            except NoSuchElementException:
                print(f"  Speech {count + 1} done.")
                uploaded = True

    input("All speeches submitted. Press Enter to close the browser ...")


# ── Run ───────────────────────────────────────────────────────────────────────
# Uncomment the lines below when the server is reachable and you are ready to run.
# check_ping()
# run_cohmetrix(trump_2020["Main text"].tolist())

## Step 3 — Merge CohMetrix Output with Speech Dataset

Reads all `CohMetrixOutput (N).txt` files produced by the web tool in numbered order (0, 1, 2, …),
transposes each into a single row of metrics, and concatenates them.

**Important:** the merge is positional — file N corresponds to speech N in the sorted `trump_2020`
DataFrame. If the filter or sort order changes the CohMetrix files must be re-generated.

In [ ]:
def import_cohmetrix_folder(folder_path: str) -> pd.DataFrame:
    """Read all CohMetrixOutput (N).txt files and return a tidy DataFrame (one row per speech)."""
    n_files = len([f for f in os.listdir(folder_path) if f.startswith("CohMetrixOutput")])
    frames  = []
    for count in range(n_files):
        path = os.path.join(folder_path, f"CohMetrixOutput ({count}).txt")
        data = pd.read_csv(path, delimiter="\t", index_col=False)
        data = data.iloc[:, [1, 3]].T          # keep label + value, transpose
        data.columns = data.iloc[0]
        data = data.iloc[1:]
        frames.append(data)
    return pd.concat(frames, ignore_index=True)


def check_duplicates(df: pd.DataFrame) -> None:
    dupes = df.duplicated()
    if dupes.any():
        print("Duplicate rows found:")
        print(df[dupes])
    else:
        print("No duplicates.")


cohmetrix_df = import_cohmetrix_folder(COHMETRIX_FOLDER)
cohmetrix_df["Record Order"] = range(len(cohmetrix_df))

df_main = pd.read_csv(DATA_PATH, sep=",")
df_main["Date Time"] = pd.to_datetime(df_main["Date Time"], format="%Y-%m-%d %H:%M:%S%z")

trump_2020_sorted = df_main[
    df_main["Title"].str.contains("Press Bri")
    & df_main["Administration"].str.contains("Trump")
].sort_index().copy()

trump_2020_sorted["Record Order"] = range(len(trump_2020_sorted))

merged = pd.merge(trump_2020_sorted, cohmetrix_df, on="Record Order")
merged.to_csv(OUTPUT_MERGED, index=False)
print(f"Merged {len(merged)} rows → {OUTPUT_MERGED}")
merged.head(3)